# PromptLab v2 — Demo Notebook
**STAT 4830 · University of Pennsylvania**

This notebook demonstrates the full PromptLab v2 pipeline and compares three generations of the rewriter model.

| Model | Architecture | Val Slop | Notes |
|-------|-------------|----------|-------|
| **Baseline** | No rewriter | ~0.85 | Generic prompt → essay |
| **v1: Prompt-mode** | Prompt→Prompt | scored live | Rewrites the prompt, then generates |
| **v2: Essay pivot** | Essay→Essay | ~0.189 | Direct essay rewriting, first run |
| **v3: Final model** | Essay→Essay | **0.0998** | LR=1e-4, r=16, 77 topics |

**How the pipeline works:**
1. A generic prompt generates a detectable AI essay (high slop score)
2. The rewriter transforms that essay to evade the detector
3. SlopDetector (editlens/roberta-large) scores each version
4. Lower score = harder to detect as AI-generated

## Cell 1 — Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os

# Clone repo if not already present
if not os.path.exists('/content/promptlab-v2'):
    !git clone https://github.com/ian-lent/promptlab-v2.git /content/promptlab-v2
    print("Cloned.")
else:
    print("Repo already present.")

os.chdir('/content/promptlab-v2')
!pip install -q peft transformers groq
print("Done.")

## Cell 2 — Adapter Paths
All paths are hardcoded from the Drive scan. No configuration needed.

In [ ]:
DRIVE = "/content/drive/MyDrive/promptlab-v2-outputs"

ADAPTERS = {
    "v1_prompt_mode": {
        "path": f"{DRIVE}/2026-04-14_17-48/outputs/rewriter/t5_lora_r8/lora_adapter",
        "mode": "prompt",   # takes prompt → outputs rewritten prompt → generates essay
        "label": "v1: Prompt-mode (prompt→prompt era)",  # slop scored live
        "color": "#EE9B00",
    },
    "v2_essay_pivot": {
        "path": f"{DRIVE}/2026-04-15_04-40_essay_mode_exp_A/outputs/rewriter/t5_lora_r8/lora_adapter",
        "mode": "essay",    # takes essay → outputs rewritten essay directly
        "label": "v2: Essay pivot (~0.189)",
        "color": "#0A9396",
    },
    "v3_final": {
        "path": f"{DRIVE}/2026-04-19_16-47_FINAL_MODEL/lora_adapter/lora_adapter",
        "mode": "essay",
        "label": "v3: Final model (0.0998) ★",
        "color": "#006600",
    },
}

# Verify all paths exist
for name, cfg in ADAPTERS.items():
    exists = os.path.exists(cfg["path"])
    status = "✓" if exists else "✗ MISSING"
    print(f"{status}  {name}: {cfg['path']}")

## Cell 3 — Load SlopDetector

In [ ]:
from detector.model import SlopDetector

print("Loading SlopDetector (editlens/roberta-large)...")
detector = SlopDetector(checkpoint="pangram/editlens_roberta-large")
print("SlopDetector ready.")

def score(text):
    """Score text with SlopDetector. 0=human-like, 1=AI-like."""
    return float(detector.score(text))

## Cell 4 — Load T5 Base + All Adapters

In [ ]:
import torch, copy
from transformers import T5ForConditionalGeneration, AutoTokenizer
from peft import PeftModel

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BASE_MODEL = "t5-base"

print(f"Device: {DEVICE}")
print("Loading T5-base...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
base_model = T5ForConditionalGeneration.from_pretrained(BASE_MODEL, torch_dtype=torch.float16)
base_model = base_model.to(DEVICE)
base_model.eval()
print("T5-base loaded.")

models = {}
for name, cfg in ADAPTERS.items():
    if not os.path.exists(cfg["path"]):
        print(f"  SKIP {name} — path missing")
        continue
    print(f"  Loading {cfg['label']}...")
    # deepcopy gives each adapter its own independent copy of the base weights
    fresh_base = copy.deepcopy(base_model)
    m = PeftModel.from_pretrained(fresh_base, cfg["path"])
    m.eval()
    models[name] = m
    print(f"  ✓ {name}")

# Free the original base model — each adapter has its own copy now
del base_model
torch.cuda.empty_cache()

print(f"\nLoaded {len(models)} independent adapters.")

## Cell 5 — Groq Setup (for baseline essay generation)

In [ ]:
import os
from groq import Groq

# Set your Groq API key
GROQ_API_KEY = "REDACTED"  # ← paste key here, or set via os.environ
if not GROQ_API_KEY:
    GROQ_API_KEY = os.environ.get("GROQ_API_KEY", "")
if not GROQ_API_KEY:
    raise ValueError("Set GROQ_API_KEY above or in environment")

groq_client = Groq(api_key=GROQ_API_KEY)

def generate_essay(prompt, model="llama-3.3-70b-versatile", max_tokens=600):
    """Generate an essay from a prompt via Groq."""
    resp = groq_client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=max_tokens,
        temperature=0.7,
    )
    return resp.choices[0].message.content

print("Groq client ready.")

## Cell 6 — Rewriter Inference Helpers

In [ ]:
def rewrite_essay(model, essay_text, max_new_tokens=512):
    """Essay-to-essay rewriting (v2, v3 models)."""
    input_text = "rewrite: " + essay_text
    enc = tokenizer(
        input_text,
        return_tensors="pt",
        max_length=512,
        truncation=True,
    ).to(DEVICE)
    with torch.no_grad():
        out = model.generate(
            **enc,
            max_new_tokens=max_new_tokens,
            num_beams=4,
            do_sample=False,
        )
    return tokenizer.decode(out[0], skip_special_tokens=True)


def rewrite_prompt_then_generate(model, original_prompt, max_new_tokens=200):
    """Prompt-to-prompt rewriting (v1 model): rewrite prompt → generate essay."""
    enc = tokenizer(
        original_prompt,
        return_tensors="pt",
        max_length=256,
        truncation=True,
    ).to(DEVICE)
    with torch.no_grad():
        out = model.generate(
            **enc,
            max_new_tokens=max_new_tokens,
            num_beams=4,
            do_sample=False,
        )
    rewritten_prompt = tokenizer.decode(out[0], skip_special_tokens=True)
    essay = generate_essay(rewritten_prompt)
    return rewritten_prompt, essay


print("Inference helpers ready.")

## Cell 7 — Display Helper

In [ ]:
from IPython.display import display, HTML

def score_bar(slop):
    pct = int(slop * 100)
    bar_color = "#AE2012" if slop > 0.6 else ("#EE9B00" if slop > 0.3 else "#006600")
    return f"""
    <div style="margin:4px 0">
      <div style="background:#eee;border-radius:4px;height:18px;width:100%;">
        <div style="background:{bar_color};height:18px;width:{pct}%;border-radius:4px;
                    display:flex;align-items:center;padding-left:6px;">
          <span style="color:white;font-size:11px;font-weight:bold;">{slop:.4f}</span>
        </div>
      </div>
    </div>"""

def display_results(topic, results):
    html = f"""
    <div style="font-family:sans-serif;max-width:900px;">
      <h2 style="color:#0D1B2A;border-bottom:3px solid #0A9396;padding-bottom:8px;">
        Topic: <em>{topic}</em>
      </h2>
      <table style="width:100%;border-collapse:collapse;">
        <tr style="background:#0D1B2A;color:white;">
          <th style="padding:10px;text-align:left;width:22%">Model</th>
          <th style="padding:10px;text-align:left;width:18%">Slop Score ↓</th>
          <th style="padding:10px;text-align:left;">Essay (first 400 chars)</th>
        </tr>
    """
    baseline_slop = results[0]["slop"]
    for i, r in enumerate(results):
        bg = "#f8f9fa" if i % 2 == 0 else "white"
        improvement = ""
        if i > 0 and baseline_slop > 0:
            pct = (baseline_slop - r["slop"]) / baseline_slop * 100
            improvement = f" <span style='color:#006600;font-size:11px;'>({pct:+.0f}% vs baseline)</span>"
        extra = f"<br><span style='color:#888;font-size:11px;font-style:italic;'>{r.get('extra','')}</span>" if r.get('extra') else ""
        html += f"""
        <tr style="background:{bg};vertical-align:top;">
          <td style="padding:10px;font-weight:bold;color:{r['color']};border-right:2px solid #eee;">
            {r['label']}{extra}
          </td>
          <td style="padding:10px;border-right:2px solid #eee;">
            {score_bar(r['slop'])}{improvement}
          </td>
          <td style="padding:10px;font-size:12px;color:#333;line-height:1.5;">
            {r['essay'][:400]}{'...' if len(r['essay']) > 400 else ''}
          </td>
        </tr>"""
    html += "</table></div>"
    display(HTML(html))

print("Display helpers ready.")

## Cell 8 — Full Pipeline Demo

Change `DEMO_TOPIC` to try different topics. The pipeline:
1. Generates a baseline essay with a generic prompt → scores it
2. **v1 (prompt-mode)**: rewrites the prompt, generates new essay → scores
3. **v2 (essay pivot)**: rewrites the baseline essay directly → scores
4. **v3 (final model)**: rewrites the baseline essay directly → scores

In [ ]:
# ── Configure ──────────────────────────────────────────────────────────────
DEMO_TOPIC = DEMO_TOPIC = "the ethics of mandatory vaccination policies"
# or
#DEMO_TOPIC = "whether universal basic income would reduce poverty"
# Other good topics:
# "the moral responsibilities of wealthy nations toward poorer ones"
# "whether artificial intelligence will create or destroy more jobs"
# "the ethics of gene editing in humans"
# "climate change and individual responsibility"
# ───────────────────────────────────────────────────────────────────────────

GENERIC_PROMPT = f"Write an essay about {DEMO_TOPIC}. Be thorough and informative."

print(f"Topic: {DEMO_TOPIC}")
print("\n[1/5] Generating baseline essay...")
baseline_essay = generate_essay(GENERIC_PROMPT)
baseline_slop = score(baseline_essay)
print(f"  Baseline slop: {baseline_slop:.4f}")

results = [{
    "label": "Baseline",
    "slop": baseline_slop,
    "essay": baseline_essay,
    "color": "#AE2012",
    "extra": "Generic prompt, no rewriter",
}]

# v1: Prompt-mode rewriter
if "v1_prompt_mode" in models:
    print("[2/5] v1 prompt-mode rewriter...")
    rewritten_prompt, v1_essay = rewrite_prompt_then_generate(
        models["v1_prompt_mode"], GENERIC_PROMPT
    )
    v1_slop = score(v1_essay)
    print(f"  v1 slop: {v1_slop:.4f}")
    print(f"  Rewritten prompt: {rewritten_prompt[:120]}...")
    results.append({
        "label": ADAPTERS["v1_prompt_mode"]["label"],
        "slop": v1_slop,
        "essay": v1_essay,
        "color": ADAPTERS["v1_prompt_mode"]["color"],
        "extra": "Rewrites prompt → generates essay (2 steps from detector)",
    })

# v2: Essay pivot
if "v2_essay_pivot" in models:
    print("[3/5] v2 essay pivot rewriter...")
    v2_essay = rewrite_essay(models["v2_essay_pivot"], baseline_essay)
    v2_slop = score(v2_essay)
    print(f"  v2 slop: {v2_slop:.4f}")
    results.append({
        "label": ADAPTERS["v2_essay_pivot"]["label"],
        "slop": v2_slop,
        "essay": v2_essay,
        "color": ADAPTERS["v2_essay_pivot"]["color"],
        "extra": "Direct essay→essay rewriting (1 step from detector)",
    })

# v3: Final model
if "v3_final" in models:
    print("[4/5] v3 final model...")
    v3_essay = rewrite_essay(models["v3_final"], baseline_essay)
    v3_slop = score(v3_essay)
    print(f"  v3 slop: {v3_slop:.4f}")
    results.append({
        "label": ADAPTERS["v3_final"]["label"],
        "slop": v3_slop,
        "essay": v3_essay,
        "color": ADAPTERS["v3_final"]["color"],
        "extra": "LR=1e-4, r=16, 77 topics, 40 contrastive pairs",
    })

print("\n[5/5] Rendering results...")
display_results(DEMO_TOPIC, results)



## Cell 9 — Side-by-Side Full Essay Text
Full essay text for each model, for qualitative inspection of what changes.

In [ ]:
def display_essays(results):
    for r in results:
        html = f"""
        <div style="font-family:sans-serif;margin-bottom:24px;
                    border-left:5px solid {r['color']};padding-left:16px;">
          <h3 style="color:{r['color']};margin-bottom:4px;">{r['label']}</h3>
          <div style="background:#f0f0f0;display:inline-block;padding:4px 12px;
                      border-radius:12px;font-size:13px;margin-bottom:12px;">
            Slop score: <strong>{r['slop']:.4f}</strong>
          </div>
          <p style="color:#333;line-height:1.7;font-size:13px;
                    white-space:pre-wrap;">{r['essay']}</p>
        </div>"""
        display(HTML(html))

display_essays(results)



## Cell 10 — Batch Evaluation Across Multiple Topics
Run the final model on several topics and report mean slop.

In [ ]:
BATCH_TOPICS = [
    "the moral responsibilities of wealthy nations toward poorer ones",
    "whether artificial intelligence will create or destroy more jobs",
    "the ethics of gene editing in humans",
    "climate change and individual responsibility",
    "the impact of remote work on urban economies",
]

if "v3_final" not in models:
    print("Final model not loaded — skipping batch eval")
else:
    print(f"{'Topic':<55} {'Baseline':>10} {'v3 Rewritten':>14} {'Improvement':>12}")
    print("-" * 95)

    baseline_scores, rewritten_scores = [], []
    for topic in BATCH_TOPICS:
        prompt = f"Write an essay about {topic}. Be thorough and informative."
        base_essay = generate_essay(prompt)
        base_s = score(base_essay)

        rw_essay = rewrite_essay(models["v3_final"], base_essay)
        rw_s = score(rw_essay)

        delta = (base_s - rw_s) / base_s * 100
        baseline_scores.append(base_s)
        rewritten_scores.append(rw_s)
        print(f"{topic[:54]:<55} {base_s:>10.4f} {rw_s:>14.4f} {delta:>11.1f}%")

    print("-" * 95)
    mean_base = sum(baseline_scores) / len(baseline_scores)
    mean_rw   = sum(rewritten_scores) / len(rewritten_scores)
    mean_delta = (mean_base - mean_rw) / mean_base * 100
    print(f"{'MEAN':<55} {mean_base:>10.4f} {mean_rw:>14.4f} {mean_delta:>11.1f}%")

## Cell 11 — Progress Chart: Slop Reduction Across Model Versions

In [ ]:
import matplotlib.pyplot as plt

# Historical val slop_mean values from the experimental record
history = [
    ("Baseline\n(no rewriter)",        0.85,  "#AE2012"),
    ("v1: Prompt-mode\n(prompt era)",  0.391, "#EE9B00"),  # approx from exp record
    ("v2: Essay pivot\n(11 pairs)",    0.189, "#0A9396"),
    ("v2: Essay mode\n(21 pairs)",     0.247, "#0A9396"),
    ("v2: LR=3e-4\n(40 pairs)",        0.196, "#94D2BD"),
    ("v3: Final model\nLR=1e-4, r=16", 0.0998,"#006600"),
]

labels = [h[0] for h in history]
values = [h[1] for h in history]
colors = [h[2] for h in history]

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(range(len(history)), values, color=colors, edgecolor="white", linewidth=1.5)

for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f"{val:.4f}", ha="center", va="bottom", fontsize=11, fontweight="bold")

ax.axhline(0.5, color="gray", linestyle="--", alpha=0.5, label="Detector threshold (0.5)")
ax.set_xticks(range(len(history)))
ax.set_xticklabels(labels, fontsize=10)
ax.set_ylabel("Mean Slop Score (lower = harder to detect)", fontsize=12)
ax.set_title("PromptLab v2: Model Progression", fontsize=14, fontweight="bold", pad=15)
ax.set_ylim(0, 1.0)
ax.spines[["top", "right"]].set_visible(False)

ax.annotate("", xy=(2, 0.189), xytext=(1, 0.391),
            arrowprops=dict(arrowstyle="->", color="#006600", lw=2))
ax.text(1.5, 0.31, "−52%\nArchitecture\npivot", ha="center", fontsize=9,
        color="#006600", fontweight="bold")

ax.annotate("", xy=(5, 0.0998), xytext=(4, 0.196),
            arrowprops=dict(arrowstyle="->", color="#006600", lw=2))
ax.text(4.5, 0.16, "−49%\nLR tuning", ha="center", fontsize=9,
        color="#006600", fontweight="bold")

plt.legend(fontsize=10)
plt.tight_layout()
plt.savefig("promptlab_v2_progress.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved promptlab_v2_progress.png")